# **Goal**

The purpose of this notebook is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

###**Pythia Model Family**

- Pythia models were originally designed for interpretability research.

- All Pythia models were trained on the same data in the same order.

- Training data: All Pythia models are trained on The Pile. The Pile is a 825GiB general-purpose dataset in English. It contains texts from 22 diverse sources, roughly broken down into five categories: academic writing (e.g. arXiv), internet (e.g. CommonCrawl), prose (e.g. Project Gutenberg), dialogue (e.g. YouTube subtitles), and miscellaneous (e.g. GitHub, Enron Emails).

- Each model saw 299,892,736,000 ~= 300B tokens during training.

####**Pythia 160M**

- <u>Total params</u>: 162,322,944. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 85,056,000. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 12
- <u>Model dimension</u>: 768. I.e.: Every token is represented internally by a vector of length 768.
- <u>MLP units per layer:</u> 3072
- <u>Heads</u>: 12
- <u>Vocabulary size:</u> 50,304. Amount of tokens in vocabulary.
- <u>Max context window</u>: 2048 tokens.

Load model

In [1]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

model = GPTNeoXForCausalLM.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step143000",
  cache_dir="./pythia-160m-deduped/step143000",
)

tokenizer = AutoTokenizer.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step143000",
  cache_dir="./pythia-160m-deduped/step143000",
)

inputs = tokenizer("Hello, I am", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  649MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'Hello, I am a newbie to the web and I have been trying to get my head around the concept of a'

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [6]:
inputs = tokenizer("How much do you like ice cream?", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=28) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'How much do you like ice cream?\n\nI love ice cream. I love the ice cream. I love the ice cream. I'

In [9]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'The capital of France is the city of Paris, and the capital of the world.\n\nThe French capital is the capital'

In [17]:

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(output[0])
print(len(output[0]))
print()
print(tokenizer.decode(output[0]))

tensor([  510,  5347,   273,  6181,   310,   253,  2639, 21601,  1321, 12169,
          323,  5489,    15,  1244,   275,  7785,    13,   835,   253,   760,
         9928, 13952,   310,   253,   378, 45117,  1079,   273,  7785,    13,
          253,  2639, 21601,  1321, 12169,   323,  5489,   310,   253,  5347,
          273,  7785,    15,  1707,   310,   671,   247,  1180,   273,   643,
         5213, 18403,   285,  4969,  2279])
55

The capital of France is the Carpentier Centre for Research. And in Paris, where the only permanent airport is the Biennale of Paris, the Carpentier Centre for Research is the capital of Paris. There is also a number of other international universities and institutes


In [44]:
inputs = tokenizer("What is love?", return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Inputs:tensor([1276,  310, 2389,   32])
Outputs: tensor([  380, 12553,  4736,   323,  1016,   285,  1046,   581,   273,   441,
           13,  1880,   359,   403,   247,  5938,   390,  4475,   281,  2656,
         1905,   476,   352,   320,  1119,   275,   253,  6203,   347,   754,
        12957, 48486,   313,  7836,    15,  2516,  1283,    27,    22,  6177,
          187,   510,  5083,  1364,  1900,  1705,   562,     2,   844,   923])

What is love? The ultimate goal for each and every one of us, whether we are a husband or wife to God — can it be found in the Lord as He reveals Himself (cf. John 18:5)?
The truth must always come out! We see


Here we are just printing the internal shape of one layer of the model.

In [38]:
print(model.gpt_neox.layers[0].mlp)

GPTNeoXMLP(
  (dense_h_to_4h): Linear(in_features=768, out_features=3072, bias=True)
  (dense_4h_to_h): Linear(in_features=3072, out_features=768, bias=True)
  (act): GELUActivation()
)


In [39]:
model.config.intermediate_size

3072

The input vector representation is 768-dimensional. The number of MLP units in the layer is 3072. Each of these produces a scalar from the input through learned weights and bises, so the layer produces a transformed and expanded 3072-d vector. After activation using GELU (Gaussian Error Linear Unit), these values are transformed back into a new 768-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{GELU}(x) = x \cdot \Phi(x) = 0.5x \left(1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right)$, where $Φ(x)$ is the cumulative distribution function of the standard normal distribution, and $\text{erf}(x)$ is the Gauss error function.

**Ablation Experiments**

In [48]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # Load weights in layer layer_idx
    mlp = model.gpt_neox.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    # Indexes in [0, 3071]
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    #
    def ablation_hook(module, inputs, output):
        indices_device = indices.to(output.device)
        output = output.clone()
        output[..., indices_device] = 0
        return output

    handle = mlp.act.register_forward_hook(ablation_hook)

    return handle, indices, layer_idx

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [71]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  2846,    13,   285,   352,   556,   247,  3072,   326,  3797,
          670,   337,    15,    22,  3041,   952,   313, 10383,   374,     6,
          275,  7785,   481,   187,   688,  1635,   281,   697,  1416,   347,
          271,  1774,  2245,  3874,   342,  1142, 15114,   824,  1284,   339,
          420,  1162,   330,   483,   265,   372,   826,   657,  4002,   390])

Zeroed activations in 0.0% of units in layer 0
The capital of France is the city, and it has a population that includes about 1.5 million people (about 2% in Paris).
In addition to its name as an important port town with many restaurants such Assement et Coutes de la Ville or


$25\%$ of first layer

In [72]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253,  954, 1774,   13,  275,  512, 2892,   15,  187,  688, 1722, 2511,
         352,  369,  247, 5112, 3874,  285,  574,  644,  271, 4383, 2846,  323,
        1142, 1107, 1078,  326,   28,  533,  672,  359,  452, 2326,  436, 2586,
         347,  581, 1270, 5674,  342,  697, 1211, 1416,  390, 4466,  281,  320,
         594, 1027])

Zeroed activations in 25.0% of units in layer 0
The capital of France is the most important, in all history.
In 1789 it was a French town and had been an English city for many years before that; but when we have seen this country as one great nation with its own name or culture to be so different


$50\%$ of first layer

In [73]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253,  954, 1774,  285,   13,   15,  187,  380, 5112,  403,  247, 1270,
        5674,  275,  436, 1388,   28,  597,  452,  616, 1211, 4454,  265,  281,
         731,  326,  359,   70,  952,  390, 1841,  347,  352,  369, 1925,  407,
         441,  432,  697, 1416,  534,  320,  323,  752,  556,  644,  594, 1048,
         264,  342])

Zeroed activations in 50.0% of units in layer 0
The capital of France is the most important and,.
 The French are a great nation in this day; they have their own nameses to them that wee people or things as it was called by us from its name which be for what has been so longed with


$75\%$ of first layer

In [74]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,   423,    13,    15,    28,   313,    27,    14,   285,   407,
          428,    16,     3,   186,   996, 50276,    64, 50275,     9,   535,
         2490,   209,     8, 50270,     4, 50254,    29,   544,  1040,    84,
          275,   253,   247,   281,   436,   260,   382,    68,   280,   336,
          261,    70,   269,    71,   453,   567,  4195,   704,  6706,  7574])

Zeroed activations in 75.0% of units in layer 0
The capital of France is
end,.; (:- and by -/"	
	  _   (

 
 '        #                        < [ |s in the a to this cistcicceise fffeff fa fe af fort


$100\%$ of first layer

In [75]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,  1018,    15,    13,     3,    27,   407,    14,  1051,  1566,
          285,  2867,  4376,   591,    16,   275,  2245,  1180,  1533,  6215,
         1612,   535,   247,   941,  2180,   313,   996,    28,   432,   575,
          544,    10,   281,   323,  1386,  1491,  1755,   367, 16832,  2216,
         3148,  2022,   186, 12275,  3601,  2317,  2460,   268,  2264, 12801])

Zeroed activations in 100.0% of units in layer 0
The capital of France is
 char.,": by-... model and property price per/ in port number world ship power

 a data period (
	; from  [) to for line information top P calendar design display main	 pixel screen space image p total orbit
